# Generic streaming batch statistics

`cx.tl.batch_process` applies a mergeable statistic within experimental batches without loading the complete cell-by-gene matrix. This example computes a batch-corrected standard deviation for every perturbation and gene.

In [1]:
from dataclasses import replace
from pathlib import Path
import sys

import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp

ROOT = Path('../..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
import crispyx as cx

OUTPUT_DIR = ROOT / 'docs' / 'notebooks' / 'tutorial_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Create a batched example

The batches deliberately have different means and variances. Computing variability within each batch prevents shifts in batch means from inflating the result.

In [2]:
rng = np.random.default_rng(7)
rows, perturbations, batches = [], [], []
for batch_index, batch in enumerate(['lane_1', 'lane_2', 'lane_3']):
    for group_index, perturbation in enumerate(['control', 'KO_A', 'KO_B']):
        n_cells = 20 + 4 * batch_index + 2 * group_index
        block = rng.normal(
            loc=4 * batch_index + group_index,
            scale=0.5 + batch_index + 0.25 * group_index,
            size=(n_cells, 6),
        )
        rows.append(block)
        perturbations.extend([perturbation] * n_cells)
        batches.extend([batch] * n_cells)

X = np.vstack(rows)
obs = pd.DataFrame(
    {'perturbation': perturbations, 'batch': batches},
    index=[f'cell_{i}' for i in range(X.shape[0])],
)
var = pd.DataFrame(index=[f'gene_{i}' for i in range(X.shape[1])])
input_path = OUTPUT_DIR / 'batch_statistics_input.h5ad'
ad.AnnData(sp.csr_matrix(X), obs=obs, var=var).write(input_path)

## Define a streaming standard-deviation reducer

A reducer is a set of small callbacks that crispyx calls while it streams the
matrix. Because it never sees all the cells at once, it has to keep running
state. The contract is:

| Callback | Receives | Must return |
| --- | --- | --- |
| `initialize(width)` | `width`: `int`, genes in this gene chunk | a fresh state object |
| `update(state, block)` | `state`, and `block`: dense `np.ndarray` of shape `(n_cells, width)` | `None` to keep the mutated state, or a replacement state |
| `finalize(state)` | `state` | `cx.BatchStatistic` whose `values` has shape `(width,)`, or a bare array of that shape |
| `compare(group_state, reference_state)` | two states from the same batch | same as `finalize`; only needed for `mode='comparison'` |

Three details matter when writing one:

- **The width is not constant.** It equals `chunk_size` for every gene chunk
  except the last, which may be shorter. Always size arrays from `width`.
- **State is per group-and-batch, and per gene chunk.** It is created fresh for
  each gene chunk, and within a chunk one state accumulates across every cell
  chunk containing that `(group, batch)` combination. `block` holds only the
  cells of that one combination, so `n_cells` differs between calls and is
  never 0.
- **The weight decides how batches combine.** crispyx reduces batches as
  `sum(weight * values) / sum(weight)` per gene. Returning the cell count as the
  weight gives a cell-count-weighted average; returning a bare array instead of a
  `BatchStatistic` is the same as weight 1 for every batch.

The reducer below tracks a count, a running mean, and a running sum of squared
deviations, so it can merge cell chunks with the parallel-variance formula and
finish with a per-batch sample standard deviation weighted by cell count.

In [3]:
def initialize_std(n_genes):
    # n_genes is the gene-chunk width; size every accumulator from it.
    # -> state: n scalar, mean (n_genes,), m2 (n_genes,)
    return {'n': 0, 'mean': np.zeros(n_genes), 'm2': np.zeros(n_genes)}

def update_std(state, block):
    # block: (n_cells, n_genes) dense, one (perturbation, batch) only.
    block_n = block.shape[0]
    block_mean = block.mean(axis=0)                      # -> (n_genes,)
    block_m2 = np.square(block - block_mean).sum(axis=0)  # -> (n_genes,)
    if state['n'] == 0:
        state.update(n=block_n, mean=block_mean, m2=block_m2)
        return                                            # None keeps this state
    # Chan et al. parallel variance: merge two (n, mean, m2) triples.
    total_n = state['n'] + block_n
    delta = block_mean - state['mean']
    state['m2'] += block_m2 + delta**2 * state['n'] * block_n / total_n
    state['mean'] += delta * block_n / total_n
    state['n'] = total_n

def finalize_std(state):
    std = np.sqrt(state['m2'] / (state['n'] - 1))  # -> (n_genes,), sample sd
    # weight=n_cells makes the cross-batch average cell-count weighted.
    return cx.BatchStatistic(std, weight=state['n'])

std_reducer = cx.BatchReducer(
    initialize=initialize_std,
    update=update_std,
    finalize=finalize_std,
)

In [4]:
std_result = cx.tl.batch_process(
    input_path,
    std_reducer,
    groupby='perturbation',  # DE-compatible alias
    batch_column='batch',
    mode='group',
    statistic_name='std',
    chunk_size=3,       # genes per chunk
    cell_chunk_size=16, # cells per reducer update
    output_path=OUTPUT_DIR / 'batch_corrected_std.h5ad',
    force=True,
)

corrected_std = pd.DataFrame(
    std_result.backed.X[:],
    index=std_result.backed.obs_names,
    columns=std_result.backed.var_names,
)
corrected_std

[cx] tl.batch_process: 3 groups × 6 genes, stratified by 'batch'
[cx] tl.batch_process: estimated disk usage: 0.0 GB (1469.9 GB free)


tl.batch_process:   0%|          | 0/2 [00:00<?, ?gene chunk/s]

Backed X is stored as CSR but is being streamed by column (axis=1) slices, which is O(total_nnz) per chunk and can be ~100x slower. Consider running cx.data.convert_to_csc(path) once to store the matrix on its fast axis.


tl.batch_process: 100%|██████████| 2/2 [00:00<00:00, 163.44gene chunk/s]

[cx] tl.batch_process: Saving → /private/tmp/crispyx/docs/notebooks/tutorial_outputs/batch_corrected_std.h5ad


,gene_0,gene_1,gene_2,gene_3,gene_4,gene_5
perturbation,,,,,,
control,1.485919,1.653347,1.677128,1.442305,1.251690,1.413554
KO_A,1.661801,1.949438,1.499869,1.914997,1.692720,1.887972
KO_B,2.181261,2.000110,2.172170,2.193765,2.152863,2.456224


For group $g$ and gene $j$, the result is $\sum_b n_{gb}s_{gbj}/\sum_b n_{gb}$: a cell-count-weighted average of within-batch sample standard deviations. The between-batch mean shift is therefore excluded.

In [5]:
reference = []
for perturbation in corrected_std.index:
    batch_stds, batch_counts = [], []
    for batch in pd.unique(obs['batch']):
        mask = (obs['perturbation'] == perturbation) & (obs['batch'] == batch)
        batch_stds.append(X[mask].std(axis=0, ddof=1))
        batch_counts.append(mask.sum())
    reference.append(np.average(batch_stds, axis=0, weights=batch_counts))

np.testing.assert_allclose(corrected_std.to_numpy(), reference)
print('Streaming and direct calculations agree.')

Streaming and direct calculations agree.


## Comparison mode

For a group-versus-reference statistic, add a `compare(group_state, reference_state)` callback and call with `mode='comparison'`. The arguments `reference`/`control_label`, `groupby`/`perturbation_column`, and `perturbations` behave like crispyx DE functions. If no reference is supplied, crispyx infers the control label using the same rules as DE. Only batches containing both the group and reference contribute.

In [6]:
def compare_means(group_state, reference_state):
    n_group = group_state['n']
    n_reference = reference_state['n']
    return cx.BatchStatistic(
        group_state['mean'] - reference_state['mean'],
        weight=n_group * n_reference / (n_group + n_reference),
    )

# BatchReducer is a frozen dataclass, so attach `compare` with dataclasses.replace
# rather than mutating the reducer defined above.
contrast_reducer = replace(std_reducer, compare=compare_means)

contrast_result = cx.tl.batch_process(
    input_path,
    contrast_reducer,
    groupby='perturbation',
    reference='control',   # omit to let crispyx infer the control label
    batch_column='batch',
    mode='comparison',
    statistic_name='mean_difference',
    chunk_size=3,
    cell_chunk_size=16,
    output_path=OUTPUT_DIR / 'batch_corrected_mean_difference.h5ad',
    force=True,
)

mean_difference = pd.DataFrame(
    contrast_result.backed.X[:],
    index=contrast_result.backed.obs_names,
    columns=contrast_result.backed.var_names,
)
mean_difference

[cx] tl.batch_process: 2 groups × 6 genes, stratified by 'batch'
[cx] tl.batch_process: estimated disk usage: 0.0 GB (1469.9 GB free)


tl.batch_process:   0%|          | 0/2 [00:00<?, ?gene chunk/s]

tl.batch_process: 100%|██████████| 2/2 [00:00<00:00, 292.44gene chunk/s]

[cx] tl.batch_process: Saving → /private/tmp/crispyx/docs/notebooks/tutorial_outputs/batch_corrected_mean_difference.h5ad


,gene_0,gene_1,gene_2,gene_3,gene_4,gene_5
perturbation,,,,,,
KO_A,0.806173,0.865520,1.386226,1.062358,0.845954,0.82975
KO_B,2.407209,1.858187,1.838216,2.538213,2.036634,1.94531


The reference group is not itself returned. For group $g$ and gene $j$ the result is $\sum_b w_{gb}(\bar{x}_{gbj} - \bar{x}_{rbj})/\sum_b w_{gb}$, where $w_{gb} = n_{gb}n_{rb}/(n_{gb} + n_{rb})$ is the harmonic weight of the group and reference cell counts in batch $b$. This is the same weighting the crispyx differential-expression functions use, and it gives batches with balanced group/reference representation the most influence.

Because the contrast is formed *within* each batch before averaging, the per-batch mean shifts built into the example data cancel out. The synthetic data was constructed so that `KO_A` sits one unit above `control` and `KO_B` two units above, in every batch. The recovered contrasts centre on 1 and 2 accordingly; the spread around those values is sampling noise from the modest number of cells per group and batch, not batch leakage.

In [7]:
reference_contrast = []
for perturbation in mean_difference.index:
    contrasts, weights = [], []
    for batch in pd.unique(obs['batch']):
        group_mask = (obs['perturbation'] == perturbation) & (obs['batch'] == batch)
        control_mask = (obs['perturbation'] == 'control') & (obs['batch'] == batch)
        n_group, n_control = group_mask.sum(), control_mask.sum()
        contrasts.append(X[group_mask].mean(axis=0) - X[control_mask].mean(axis=0))
        weights.append(n_group * n_control / (n_group + n_control))
    reference_contrast.append(np.average(contrasts, axis=0, weights=weights))

np.testing.assert_allclose(mean_difference.to_numpy(), reference_contrast)
print('Streaming and direct contrasts agree.')

Streaming and direct contrasts agree.


## Inspecting the result

`batch_process` writes a disk-backed AnnData. Alongside `X` it records the total aggregation weight per group and gene in a `weight_sum` layer, the number of batches that contributed to each group in `obs`, and the call parameters in `uns` — enough to tell a genuinely small effect apart from one estimated from very few cells.

In [8]:
backed = contrast_result.backed

print('result shape      :', backed.shape)
print('batches per group :', backed.obs['n_batches_used'].to_dict())
print('weight_sum (gene_0):', dict(zip(backed.obs_names, backed.layers['weight_sum'][:, 0])))
print()
for key in ('statistic_name', 'mode', 'perturbation_column', 'batch_column'):
    if key in backed.uns:
        print(f'uns[{key!r}] = {backed.uns[key]!r}')

result shape      : (2, 6)
batches per group : {'KO_A': 3, 'KO_B': 3}
weight_sum (gene_0): {'KO_A': np.float64(37.43894909688014), 'KO_B': np.float64(38.76550116550116)}

uns['statistic_name'] = 'mean_difference'
uns['mode'] = 'comparison'
uns['perturbation_column'] = 'perturbation'
uns['batch_column'] = 'batch'


## Multiple channels from one streaming pass

`finalize`/`compare` so far returned one statistic. A related pair that shares
the same underlying state — a mean difference and its standard error, so a
caller can form `t = mean / se` — can be packed into a **single** pass instead
of two: set `channels` on the `BatchReducer` to the tuple of names it will
return, and have `compare` return a dict with exactly those keys instead of
one bare value. Each channel is combined across batches independently (with
its own weight) and written to its own `layers[name]`; the first channel is
also copied into `X`.

To compare the result against `cx.t_test` on equal footing, this section
uses a second, unbatched dataset (`t_test` does not stratify by batch, so
comparing it against the batch-corrected contrast above would mix in a
genuine methodological difference, not just illustrate the mean/se identity).

In [9]:
rng2 = np.random.default_rng(11)
rows2, perturbations2 = [], []
for group_index, perturbation in enumerate(['control', 'KO_A', 'KO_B']):
    n_cells = 40 + 5 * group_index
    # Shared +2.0 offset (irrelevant to mean_diff/se/t, which only depend on
    # differences) keeps every group away from 0, avoiding a log2(~0) warning
    # in t_test's unrelated log-fold-change computation below.
    block = rng2.normal(loc=2.0 + 0.5 * group_index, scale=1.0, size=(n_cells, 6))
    rows2.append(block)
    perturbations2.extend([perturbation] * n_cells)

X2 = np.vstack(rows2)
obs2 = pd.DataFrame(
    # A single shared 'lane' value: batch_process still requires a batch_column,
    # but with one batch its stratified statistic reduces to the plain,
    # unstratified one -- comparable to cx.t_test, which never stratifies.
    {'perturbation': perturbations2, 'lane': 'lane_1'},
    index=[f'cell2_{i}' for i in range(X2.shape[0])],
)
var2 = pd.DataFrame(index=[f'gene_{i}' for i in range(X2.shape[1])])
mean_se_input_path = OUTPUT_DIR / 'mean_se_input.h5ad'
ad.AnnData(sp.csr_matrix(X2), obs=obs2, var=var2).write(mean_se_input_path)

In [10]:
def compare_mean_se(group_state, reference_state):
    n_group, n_reference = group_state['n'], reference_state['n']
    var_group = group_state['m2'] / (n_group - 1)
    var_reference = reference_state['m2'] / (n_reference - 1)
    mean_diff = group_state['mean'] - reference_state['mean']
    se = np.sqrt(var_group / n_group + var_reference / n_reference)
    weight = n_group * n_reference / (n_group + n_reference)
    return {
        'mean_diff': cx.BatchStatistic(mean_diff, weight=weight),
        'se': cx.BatchStatistic(se, weight=weight),
    }

# Reuse std_reducer's initialize/update (they already track n, mean, m2);
# only compare() and channels are new.
mean_se_reducer = replace(std_reducer, compare=compare_mean_se, channels=('mean_diff', 'se'))

mean_se_result = cx.tl.batch_process(
    mean_se_input_path,
    mean_se_reducer,
    groupby='perturbation',
    reference='control',
    batch_column='lane',
    mode='comparison',
    statistic_name='mean_se',
    chunk_size=3,
    cell_chunk_size=32,
    output_path=OUTPUT_DIR / 'batch_corrected_mean_se.h5ad',
    force=True,
)

backed_mean_se = mean_se_result.backed
t_from_mean_se = pd.DataFrame(
    np.asarray(backed_mean_se.layers['mean_diff'][:]) / np.asarray(backed_mean_se.layers['se'][:]),
    index=backed_mean_se.obs_names,
    columns=backed_mean_se.var_names,
)
print('channels in uns:', backed_mean_se.uns['channels'])
print('layers:', sorted(backed_mean_se.layers.keys()))
t_from_mean_se

[cx] tl.batch_process: 2 groups × 6 genes, stratified by 'lane'
[cx] tl.batch_process: estimated disk usage: 0.0 GB (1469.9 GB free)


tl.batch_process:   0%|          | 0/2 [00:00<?, ?gene chunk/s]

tl.batch_process: 100%|██████████| 2/2 [00:00<00:00, 387.00gene chunk/s]

[cx] tl.batch_process: Saving → /private/tmp/crispyx/docs/notebooks/tutorial_outputs/batch_corrected_mean_se.h5ad
channels in uns: ['mean_diff' 'se']
layers: ['mean_diff', 'mean_diff_weight_sum', 'se', 'se_weight_sum']


,gene_0,gene_1,gene_2,gene_3,gene_4,gene_5
perturbation,,,,,,
KO_A,2.170589,3.733261,1.709632,2.519494,3.751254,1.574102
KO_B,4.127171,4.760559,5.337958,4.081797,5.454586,4.188923


`mean_diff / se` above *is* a t-statistic, by construction — the same Welch's
t-statistic `cx.t_test` computes, just assembled here from two channels of
one streaming pass instead of `t_test`'s dedicated implementation. To confirm
that identity, run `cx.t_test` on the same data with its per-condition
low-expression filter disabled (so no gene is excluded on either side, making
this a fair value-for-value comparison):

In [11]:
t_test_results = cx.t_test(
    mean_se_input_path,
    perturbation_column='perturbation',
    control_label='control',
    min_cells_expressed=0,
    min_pct_ctrl=0.0,
    min_pct_pert=0.0,
    min_mean_ctrl=-np.inf,
    min_mean_pert=-np.inf,
    output_dir=OUTPUT_DIR,
    data_name='mean_se_comparison',
    verbose=False,
)

# rtol accounts for t_test's internal float32 buffers vs. this reducer's float64.
for group in ('KO_A', 'KO_B'):
    np.testing.assert_allclose(
        t_test_results[group].statistic, t_from_mean_se.loc[group].to_numpy(), rtol=1e-5,
    )
print('t = mean_diff / se from batch_process matches cx.t_test (up to float32 precision).')

t = mean_diff / se from batch_process matches cx.t_test (up to float32 precision).


**Use `cx.t_test` for real analyses.** With filtering re-enabled (the
defaults), `t_test` automatically excludes genes that are jointly
low-expressed in both the group and the control (`min_pct_ctrl`,
`min_pct_pert`, `min_mean_ctrl`, `min_mean_pert`) and reports proper
Welch–Satterthwaite p-values alongside the statistic. The `mean_se` reducer
above has none of that — it exists to demonstrate that a `BatchReducer` can
emit several related channels from a single streaming pass, not to replace
the dedicated test.